In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df_raw=pd.read_csv("flower_3class.csv")
df=df_raw.copy()

In [5]:
cols_x = ['petal_length', 'petal_width']
cols_y = 'species'
X=df[cols_x].values.astype(float)
Y=df[cols_y].values.astype(int)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [7]:
X_train_raw=X_train.copy()
X_test_raw=X_test.copy()

In [8]:
means=X_train_raw.mean(axis=0)
stds=X_train_raw.std(axis=0,ddof=0)
X_train_scaled = (X_train_raw - means) / stds
X_test_scaled = (X_test_raw - means) / stds
X_train = np.hstack([
    np.ones((X_train_scaled.shape[0], 1)),
    X_train_scaled
])

X_test = np.hstack([
    np.ones((X_test_scaled.shape[0], 1)),
    X_test_scaled
])
K = len(np.unique(Y))      
n = X_train.shape[1]
m=X_train.shape[0]

Y_train_onehot = np.eye(K)[Y_train]

def softmax(Z):
    Z = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)
    
W = np.zeros((n, K))

alpha = 0.5
n_iters = 5000

cost_history = []
grad_history = []
for i in range(n_iters):
    Z=X_train@W
    P=softmax(Z)
    cost=-np.mean(np.sum(Y_train_onehot*np.log(P+1e-15),axis=1))
    cost_history.append(cost)
    grad=(X_train.T@(P-Y_train_onehot))/m
    grad_history.append(np.linalg.norm(grad))
    W=W-alpha*grad

def predict_proba(X):
    Z = X @ W
    P = softmax(Z)
    return P

def predict(X):
    P = predict_proba(X)
    return np.argmax(P, axis=1)



train_pred = predict(X_train)
test_pred = predict(X_test)

train_acc = np.mean(train_pred == Y_train)
test_acc = np.mean(test_pred == Y_test)

print("训练集准确率:", train_acc)
print("测试集准确率:", test_acc)
print("最终损失:", cost_history[-1])
print("参数矩阵 W:")
print(W)

训练集准确率: 0.9916666666666667
测试集准确率: 0.9333333333333333
最终损失: 0.04291698678491997
参数矩阵 W:
[[ 0.36667973 -0.30239584 -0.06428389]
 [-4.52363958  5.0276347  -0.50399512]
 [-2.73504939 -4.02887229  6.76392168]]
